In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 12: Fine-tune dengan LoRA

Cerita lengkapnya ada di `README.md`. Hitung dulu, baru jalankan kode.

Mode cepat memakai numpy di CPU. Tidak ada torch, tidak ada unduhan, bukan data asli perusahaan.

Satu hitungan tangan: vektor `[1.0, 0.0]` dikali `0.5` menjadi apa?


In [2]:
# Satu ide: numpy dipakai di mode cepat, tanpa torch dan tanpa unduhan.
import numpy as np


In [3]:
# Satu ide: empat angka sintetis menjadi satu label ya atau tidak.
rng = np.random.default_rng(SEED)
X = rng.normal(size=(40, 4))
arah = np.array([0.7, -0.4, 0.5, 0.2])
y = (X @ arah > 0).astype(float)


In [4]:
# Satu ide: pecah set latih dan set uji yang sama untuk sebelum dan sesudah.
acak = rng.permutation(len(X))
X_latih, y_latih = X[acak[:30]], y[acak[:30]]
X_uji, y_uji = X[acak[30:]], y[acak[30:]]
print("baris_latih", int(len(X_latih)))
print("baris_uji", int(len(X_uji)))


baris_latih 30
baris_uji 10


In [5]:
# Satu ide: bobot dasar dibekukan, yang dilatih hanya catatan kecil.
W = rng.normal(scale=0.05, size=4)
W_awal = W.copy()
a = rng.normal(scale=0.1, size=4)
b = 0.0
print("pangkat", 1)


pangkat 1


In [6]:
# Satu ide: fungsi kerugian adalah rata-rata selisih yang dikuadratkan.
def kerugian(bobot, vektor_a, skalar_b, data_x, data_y):
    delta = bobot + skalar_b * vektor_a
    pred = 1 / (1 + np.exp(-(data_x @ delta)))
    return float(np.mean((pred - data_y) ** 2))


In [7]:
# Satu ide: ukur set uji sebelum catatan kecil dilatih.
sebelum = kerugian(W, a, b, X_uji, y_uji)
pred_awal = 1 / (1 + np.exp(-(X_uji @ (W + b * a))))
salah_sebelum = int(((pred_awal >= 0.5) != (y_uji >= 0.5)).sum())
print("kerugian_uji_sebelum", round(sebelum, 6))
print("salah_sebelum", salah_sebelum)


kerugian_uji_sebelum 0.246913
salah_sebelum 4


In [8]:
# Satu ide: delapan puluh langkah hanya menggeser catatan kecil.
for _ in range(80):
    pred = 1 / (1 + np.exp(-(X_latih @ (W + b * a))))
    d = (2 * (pred - y_latih) / len(y_latih)) * pred * (1 - pred)
    g = X_latih.T @ d
    a_baru = a - 0.8 * (b * g)
    b = b - 0.8 * float(a @ g)
    a = a_baru


In [9]:
# Satu ide: set uji yang sama diukur lagi sesudah catatan dilatih.
sesudah = kerugian(W, a, b, X_uji, y_uji)
pred_akhir = 1 / (1 + np.exp(-(X_uji @ (W + b * a))))
salah_sesudah = int(((pred_akhir >= 0.5) != (y_uji >= 0.5)).sum())
selisih = sebelum - sesudah
print("kerugian_uji_sesudah", round(sesudah, 6))
print("selisih_uji", round(selisih, 6))
print("salah_sesudah", salah_sesudah)
assert selisih >= 0.15
assert salah_sesudah < salah_sebelum


kerugian_uji_sesudah 0.050404
selisih_uji 0.196509
salah_sesudah 1


In [10]:
# Satu ide: bobot dasar tidak berubah dan catatan tidak disimpan ke berkas.
tetap = bool(np.allclose(W, W_awal))
print("bobot_tetap", "ya" if tetap else "tidak")
print("adapter_disimpan", "tidak")
assert tetap


bobot_tetap ya
adapter_disimpan tidak


In [11]:
# Satu ide: mode penuh memakai revisi yang dipin, mode cepat tidak mengunduh.
import importlib.util
import sys
from pathlib import Path
akar = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "phase2-ai" / "muat_hf.py").is_file())
sys.path.insert(0, str(akar / "phase2-ai"))
from muat_hf import REVISI
spec = importlib.util.spec_from_file_location("muat12", akar / "phase2-ai" / "modul-12-lora" / "muat.py")
muat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(muat)
print("model", muat.ID_CEPAT)
print("revisi", REVISI[muat.ID_CEPAT])
print("model_penuh", muat.ID_PENUH)
print("revisi_penuh", REVISI[muat.ID_PENUH])
print("unduh", False)


model HuggingFaceTB/SmolLM2-135M-Instruct
revisi 12fd25f77366fa6b3b4b768ec3050bf629380bac
model_penuh Qwen/Qwen2.5-0.5B-Instruct
revisi_penuh 7ae557604adf67be50417f59c2c2f167def9a775
unduh False


## Jalur LoRA nyata

Sel di bawah jalan bila `A2N_TORCH=1` dan grup torch sudah terpasang. Jika tidak, sel mencetak kalimat lewati. Perintah pasangnya ada di README.


In [12]:
# Satu ide: jalur LoRA nyata hanya bila A2N_TORCH=1 dan grup torch terpasang.
import importlib.util
import os
ada_torch = importlib.util.find_spec("torch") is not None
ada_peft = importlib.util.find_spec("peft") is not None
jalur_lora = os.environ.get("A2N_TORCH") == "1" and ada_torch and ada_peft
PESAN = "Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1."
print("jalur_lora", "nyala" if jalur_lora else "lewati")


jalur_lora lewati


In [13]:
# Satu ide: jaringan torch kecil dibekukan sebelum catatan LoRA dipasang.
if not jalur_lora:
    print("lewati_beku", PESAN)
else:
    import torch
    from torch import nn
    class Jaring(nn.Module):
        def __init__(self):
            super().__init__()
            self.lapisan = nn.Linear(4, 1, bias=False)
        def forward(self, x):
            return self.lapisan(x)
    torch.manual_seed(SEED)
    jaring = Jaring()
    print("beku", "siap")


lewati_beku Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [14]:
# Satu ide: peft menempelkan LoRA pada lapisan yang sudah dibekukan.
if not jalur_lora:
    print("lewati_peft", PESAN)
else:
    from peft import LoraConfig, inject_adapter_in_model
    for nilai in jaring.parameters():
        nilai.requires_grad_(False)
    konfigurasi = LoraConfig(
        r=1, lora_alpha=1, target_modules=["lapisan"], bias="none"
    )
    jaring = inject_adapter_in_model(konfigurasi, jaring)
    print("adapter_peft", "ya")


lewati_peft Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [15]:
# Satu ide: latih hanya parameter LoRA pada data sintetis yang sama.
if not jalur_lora:
    print("lewati_latih_peft", PESAN)
else:
    xb = torch.tensor(X_latih, dtype=torch.float32)
    yb = torch.tensor(y_latih, dtype=torch.float32).view(-1, 1)
    opt = torch.optim.SGD([p for p in jaring.parameters() if p.requires_grad], lr=0.8)
    opt.zero_grad()
    awal_peft = float(torch.mean((torch.sigmoid(jaring(xb)) - yb) ** 2).detach())
    for _ in range(80):
        opt.zero_grad()
        loss_p = torch.mean((torch.sigmoid(jaring(xb)) - yb) ** 2)
        loss_p.backward()
        opt.step()


lewati_latih_peft Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [16]:
# Satu ide: fungsi kerugian pada data latih turun setelah LoRA nyata.
if not jalur_lora:
    print("lewati_hasil_peft", PESAN)
else:
    turun_peft = awal_peft - float(loss_p.detach())
    print("turun_peft", "ya" if turun_peft > 0 else "tidak")
    assert turun_peft > 0


lewati_hasil_peft Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.
